# Model 2: GP correction of the GC error, RQ kernel

**Purpose.** Fit one GC-GP model form to all seven properties and score it with 10-fold
cross-validation. All twelve models use the same folds, so this is one of the twelve
models compared in the kernel study (summarized in `03_Kernel_Comparison.ipynb`).

**Model form (method 2):** `y = y_GC + GP(0, K(MW))`

The GP learns the *error* of the GC estimate (experimental minus GC) from the molecular weight alone, and the prediction is the GC estimate plus this learned correction. Inputs and targets are not standardized for this method.

**What this notebook produces** (in `Output_Data/kernel_comparison/cv/model_2/`)

* `<property>/<property>_cv_summary.json`: pooled and per-fold metrics (read by notebook 03).
* `<property>/<property>_fold_metrics.csv`: train and test metrics of every fold.
* `<property>/<property>_all_cv_predictions.csv`: every prediction with its 95% interval, fold and set.
* `<property>/<property>_GP_CV_parity.png` / `.pdf` and `<property>_GP_CV_fold_r2.png` / `.pdf`: parity and per-fold plots (600 dpi).
* `<property>/fold_<k>_model_summary.txt`: fitted hyperparameters of every fold.
* `summary.csv`, `test_parity_overview.png` and `results_summary.md`: the overview of this model.

## 1. Environment and setup

In [ ]:
import sys
from pathlib import Path

# Find the repository root (the folder that contains src/gcgp) and make the
# shared code importable. This works whether Jupyter starts in notebooks/ or
# in the repository root.
repoRoot = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gcgp").is_dir())
sys.path.insert(0, str(repoRoot / "src"))

import pandas as pd

from gcgp.settings import checkCuratedData, describeSetup, outputFolder, setGlobalSeed
from gcgp.kernelCv import compactCvSummary, cvFolder, cvTestPredictions, runCvPlan
from gcgp.plotting import plotTestParityGrid
from gcgp.checkpoints import checkpointStatus

setGlobalSeed()
outputDir = outputFolder("kernel_comparison")
pd.set_option("display.precision", 3)
checkCuratedData()
print(describeSetup())
print(checkpointStatus())

**Checkpoints.** Every GP optimizer restart is saved in
`Output_Data/checkpoints/` as soon as it finishes. If the notebook stops (or is
rerun), finished restarts are loaded instead of refitted, so the run resumes
where it stopped and gives the same numbers. Set the environment variable
`GCGP_RESUME=0` to ignore the saved restarts and refit everything.

**Parallel restarts.** Restarts that still have to be fitted can run in
parallel worker processes: set the environment variable `GCGP_N_WORKERS`
(for example 4) before starting the kernel, or use
`bash scripts/run_notebooks.sh --workers 4`. The results are bit-for-bit the
same for any number of workers; `scripts/benchmark_gp_workers.py` shows which
number is fastest on your computer.

## 2. Data and model settings

Each property is read from its curated dataset in `Output_Data/curated_data/`
(written by `00_Data_Curation.ipynb`). The inputs are molecular weight and the
GC estimate; the target is the CRC value.

Every entry of the run plan below lists the properties it applies to and the GP
settings used for them:

* `kernel`: covariance function (RQ, RBF, or a Matérn kernel).
* `anisotropic`: whether each input gets its own length scale.
* `optMethod`: SciPy optimizer for the hyperparameters.
* `trainLikelihood`: whether the noise term is fitted (if False it stays at 1e-5).
* `retrainGp`: number of random restarts; the best converged restart is kept.

**Notes for this model.** Same settings for every property.

In [ ]:
methodNumber = 2
saveName = "model_2"

runPlan = [
    {"properties": ["Tb", "Tm", "Pc", "Vc", "Tc", "Hvap", "logP"],
     "kernel": "RQ", "anisotropic": False, "optMethod": "L-BFGS-B",
     "trainLikelihood": False, "retrainGp": 10},
]

pd.DataFrame(runPlan)

## 3. Cross-validate

For each property: cut the molecules into 10 folds (shuffled, seed 42), hold each
fold out once, fit the GP with several restarts on the other nine, and predict the
held-out fold. The folds depend only on the data and the seed, so every model of the
study is tested on the same molecules in each fold. A fold whose GP cannot be fitted
is dropped and reported; the property is left out only when fewer than five folds
succeed. The per-property figures are saved to disk rather than shown here, to keep
the notebook short; Section 4 shows an overview.


In [ ]:
summary = runCvPlan(runPlan, methodNumber, saveName, outputDir)
modelDir = cvFolder(outputDir, saveName)
summary.to_csv(modelDir / "summary.csv", index=False)

## 4. Results

### 4.1 Metrics for every run

`Overall_Test_R2`, `Overall_Test_RMSE` and `Overall_Test_MAE` pool the out-of-fold
predictions, so every molecule is predicted once by a GP that never saw it.
`Mean_Test_R2 ± Std_Test_R2` shows how much R² varies from fold to fold.
`CI_Test_Outside_Frac` is the fraction of molecules whose experimental value falls
outside the predicted 95% interval; a well-calibrated model should be close to 0.05.


In [ ]:
compactCvSummary(summary)

### 4.2 Out-of-fold parity plots

Each panel shows the out-of-fold predictions for one property. The error bars are the
95% prediction intervals and the dashed line is perfect agreement.


In [ ]:
tables = {code: cvTestPredictions(outputDir, saveName, code) for code in summary["Property"]}
if tables:
    plotTestParityGrid(tables, f"Model {2}, out-of-fold predictions",
                       savePath=modelDir / "test_parity_overview.png")

## 5. Summary of the results

The text below is written from the metrics of this run and saved as
`results_summary.md` in this model's output folder. A fold that could not be
fitted (for example when no optimizer restart converged) is dropped and reported
instead of stopping the notebook. The summary files are read by
`03_Kernel_Comparison.ipynb`, which puts all twelve models side by side.

In [ ]:
from gcgp.narrative import describeKernelCv, showAndSave

showAndSave(describeKernelCv(summary, "Model 2: GP correction of the GC error, RQ kernel"),
            modelDir / "results_summary.md");